# HW04 Part B - Text Task: NSMC LSTM/TextCNN

이 노트북은 과제 제출용 baseline 코드이다. 위에서부터 순서대로 실행하면 데이터 로드, 모델 학습, 성능 저장, loss/accuracy 그래프 저장까지 수행한다.

## 실행 전 확인
- GPU 런타임 사용 권장
- 결과는 `./outputs` 폴더에 저장됨
- 최종 보고서에는 `*_metrics_summary.csv`, `*_loss_curve.png` 값을 반영하면 됨

In [ ]:
"""
Week 10 HW04 - Part B. Text Task
Dataset: NSMC Korean movie review sentiment classification
Models: LSTM baseline + optional Text CNN baseline
Outputs:
  - outputs/taskB_metrics_summary.csv
  - outputs/taskB_history.csv
  - outputs/taskB_loss_curve.png
  - outputs/taskB_accuracy_curve.png

Run:
  python HW04_TaskB_Text_NSMC_LSTM_TextCNN.py

Tokenization used in this file:
  - Regular-expression based Korean/English/number tokenization
  - Lowercasing English text
  - Removing punctuation and splitting by whitespace
  - No morphological analyzer is required, so this runs easily in Colab/local Python.

If your class requires morphological analysis, replace basic_korean_tokenizer()
with a KoNLPy/Mecab tokenizer and keep the rest of the pipeline unchanged.
"""

import os
import random
import re
import urllib.request
from collections import Counter
from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split


# =========================================================
# 0. Config
# =========================================================
SEED = 42
BATCH_SIZE = 128
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
VALID_RATIO = 0.1
MAX_LEN = 80
MAX_VOCAB_SIZE = 30000
MIN_FREQ = 2
EMBED_DIM = 128
HIDDEN_DIM = 128
RUN_TEXTCNN = True

# Set to None for full NSMC. Smaller values make homework runs faster.
MAX_TRAIN_SAMPLES = 50000
MAX_TEST_SAMPLES = 10000

DATA_DIR = Path("./data/nsmc")
OUT_DIR = Path("./outputs")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")


# =========================================================
# 1. Download / Load NSMC
# =========================================================
def download_if_needed():
    urls = {
        "ratings_train.txt": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
        "ratings_test.txt": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    }
    for filename, url in urls.items():
        path = DATA_DIR / filename
        if path.exists():
            continue
        print(f"[INFO] Downloading {filename} ...")
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            raise RuntimeError(
                f"Failed to download {filename}. "
                f"Please manually place it in {DATA_DIR}. Original error: {e}"
            )


def load_nsmc() -> Tuple[pd.DataFrame, pd.DataFrame]:
    download_if_needed()
    train_df = pd.read_csv(DATA_DIR / "ratings_train.txt", sep="\t")
    test_df = pd.read_csv(DATA_DIR / "ratings_test.txt", sep="\t")

    train_df = train_df.dropna(subset=["document", "label"]).drop_duplicates(subset=["document"])
    test_df = test_df.dropna(subset=["document", "label"]).drop_duplicates(subset=["document"])

    if MAX_TRAIN_SAMPLES is not None:
        train_df = train_df.sample(n=min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED)
    if MAX_TEST_SAMPLES is not None:
        test_df = test_df.sample(n=min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED)

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    return train_df, test_df


train_df, test_df = load_nsmc()
print(f"[INFO] NSMC train/test documents: {len(train_df)}/{len(test_df)}")
print(f"[INFO] Classes: {sorted(train_df['label'].unique().tolist())} (0=negative, 1=positive)")


# =========================================================
# 2. Tokenizer / Vocabulary
# =========================================================
def basic_korean_tokenizer(text: str) -> List[str]:
    """
    Simple tokenizer for Korean review baseline.
    - Keeps Korean syllables/Jamo, English letters, numbers, and spaces.
    - Removes punctuation.
    - Splits by whitespace.

    This is not a perfect Korean morphological tokenizer, but it is sufficient
    for a lightweight baseline and is easy to run without external Java/Mecab setup.
    """
    text = str(text).lower()
    text = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9\s]", " ", text)
    tokens = text.split()
    return tokens


def build_vocab(texts: List[str], max_vocab_size: int, min_freq: int) -> Dict[str, int]:
    counter = Counter()
    for text in texts:
        counter.update(basic_korean_tokenizer(text))

    # 0: PAD, 1: UNK
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.most_common(max_vocab_size - 2):
        if freq < min_freq:
            break
        vocab[word] = len(vocab)
    return vocab


def encode_text(text: str, vocab: Dict[str, int], max_len: int) -> List[int]:
    tokens = basic_korean_tokenizer(text)
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids


vocab = build_vocab(train_df["document"].tolist(), MAX_VOCAB_SIZE, MIN_FREQ)
print(f"[INFO] Vocab size: {len(vocab)}")


# =========================================================
# 3. Dataset / DataLoader
# =========================================================
class NSMCDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: Dict[str, int], max_len: int):
        self.input_ids = [encode_text(text, vocab, max_len) for text in df["document"].tolist()]
        self.labels = df["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.input_ids[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


full_train_dataset = NSMCDataset(train_df, vocab, MAX_LEN)
test_dataset = NSMCDataset(test_df, vocab, MAX_LEN)

val_size = int(len(full_train_dataset) * VALID_RATIO)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"[INFO] Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")


# =========================================================
# 4. Models
# =========================================================
class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_classes: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)  # (B, L, E)
        _, (h_n, _) = self.lstm(embedded)
        # bidirectional, num_layers=1: h_n[-2] forward final, h_n[-1] backward final
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        h = self.dropout(h)
        return self.fc(h)


class TextCNNClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_classes: int = 2, num_filters: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=0)
            for k in [3, 4, 5]
        ])
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(num_filters * 3, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)          # (B, L, E)
        x = x.permute(0, 2, 1)                # (B, E, L)
        pooled_outputs = []
        for conv in self.convs:
            z = torch.relu(conv(x))           # (B, F, L-k+1)
            z = torch.max(z, dim=2).values    # global max pooling, (B, F)
            pooled_outputs.append(z)
        h = torch.cat(pooled_outputs, dim=1)
        h = self.dropout(h)
        return self.fc(h)


# =========================================================
# 5. Train / Evaluation functions
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for input_ids, labels in loader:
        input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for input_ids, labels in loader:
        input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)
        logits = model(input_ids)
        loss = criterion(logits, labels)

        total_loss += loss.item() * input_ids.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def run_experiment(model_name: str, model: nn.Module):
    print(f"\n========== Training {model_name} ==========")
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = []
    best_val_acc = 0.0
    best_path = OUT_DIR / f"taskB_{model_name}_best.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        row = {
            "model": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
        )

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    summary = {
        "model": model_name,
        "best_val_acc": best_val_acc,
        "test_loss": test_loss,
        "test_acc": test_acc,
    }
    print(f"[TEST] {model_name}: loss={test_loss:.4f}, acc={test_acc:.4f}")
    return pd.DataFrame(history), summary


# =========================================================
# 6. Run LSTM and optional TextCNN
# =========================================================
histories = []
summaries = []

lstm_history, lstm_summary = run_experiment(
    "LSTM",
    LSTMSentimentClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM),
)
histories.append(lstm_history)
summaries.append(lstm_summary)

if RUN_TEXTCNN:
    textcnn_history, textcnn_summary = run_experiment(
        "TextCNN",
        TextCNNClassifier(len(vocab), EMBED_DIM),
    )
    histories.append(textcnn_history)
    summaries.append(textcnn_summary)

history_df = pd.concat(histories, ignore_index=True)
summary_df = pd.DataFrame(summaries)

history_df.to_csv(OUT_DIR / "taskB_history.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv(OUT_DIR / "taskB_metrics_summary.csv", index=False, encoding="utf-8-sig")
print("\n[INFO] Saved metrics CSV files in ./outputs")
print(summary_df)


# =========================================================
# 7. Plot loss and accuracy curves
# =========================================================
def plot_curves(history: pd.DataFrame, metric: str, ylabel: str, out_path: Path):
    plt.figure(figsize=(8, 5))
    for model_name in history["model"].unique():
        sub = history[history["model"] == model_name]
        plt.plot(sub["epoch"], sub[f"train_{metric}"], marker="o", label=f"{model_name} train")
        plt.plot(sub["epoch"], sub[f"val_{metric}"], marker="s", linestyle="--", label=f"{model_name} val")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(f"Part B Text Task - {ylabel} Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


plot_curves(history_df, "loss", "Loss", OUT_DIR / "taskB_loss_curve.png")
plot_curves(history_df, "acc", "Accuracy", OUT_DIR / "taskB_accuracy_curve.png")
print("[INFO] Saved plots: taskB_loss_curve.png, taskB_accuracy_curve.png")
